# OpenPFLOW data overview

`trajectory01.tsv.zip` から `trajectory06.tsv.zip` までを読み込み、データ構造と分布をざっと確認するためのノートです。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)
sns.set_theme(style="whitegrid")

In [ ]:
data_dir = Path("./data")
zip_paths = sorted(data_dir.glob("trajectory*.tsv.zip"))

zip_paths

In [ ]:
COLUMN_NAMES = ["id", "timestamp", "lon", "lat", "transport_mode"]


def read_trajectory_zip(zip_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_df = pd.read_csv(
        zip_path,
        sep="\t",
        header=None,
        names=COLUMN_NAMES,
        dtype="string",
        compression="zip",
        on_bad_lines="skip",
    )

    df = raw_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["transport_mode"] = pd.to_numeric(df["transport_mode"], errors="coerce").astype("Int64")
    df["source_file"] = zip_path.name

    # 先頭に壊れたレコードが混じるので、最低限の時空間カラムが揃う行だけ残す。
    valid_mask = df[["timestamp", "lon", "lat"]].notna().all(axis=1)
    clean_df = df.loc[valid_mask].reset_index(drop=True)

    return raw_df, clean_df

In [ ]:
raw_dfs = {}
clean_dfs = {}
summary_rows = []

for zip_path in zip_paths:
    raw_df, clean_df = read_trajectory_zip(zip_path)
    raw_dfs[zip_path.name] = raw_df
    clean_dfs[zip_path.name] = clean_df

    summary_rows.append(
        {
            "file": zip_path.name,
            "raw_rows": len(raw_df),
            "valid_rows": len(clean_df),
            "dropped_rows": len(raw_df) - len(clean_df),
            "n_ids": clean_df["id"].nunique(),
            "time_min": clean_df["timestamp"].min(),
            "time_max": clean_df["timestamp"].max(),
            "lon_min": clean_df["lon"].min(),
            "lon_max": clean_df["lon"].max(),
            "lat_min": clean_df["lat"].min(),
            "lat_max": clean_df["lat"].max(),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
for file_name in clean_dfs:
    print(f"=== {file_name} ===")
    print("raw preview")
    display(raw_dfs[file_name].head())
    print("clean preview")
    display(clean_dfs[file_name].head())

In [ ]:
df = pd.concat(clean_dfs.values(), ignore_index=True)
df = df.sort_values(["timestamp", "id"]).reset_index(drop=True)
df["hour"] = df["timestamp"].dt.hour
df["date"] = df["timestamp"].dt.date

display(df.head())
display(df.describe(include="all").T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

summary_df.set_index("file")["valid_rows"].plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Valid rows by file")
axes[0].set_ylabel("rows")
axes[0].tick_params(axis="x", rotation=45)

mode_counts = df["transport_mode"].value_counts().sort_index()
mode_counts.plot(kind="bar", ax=axes[1], color="#F58518")
axes[1].set_title("Transport mode counts")
axes[1].set_xlabel("transport_mode")
axes[1].set_ylabel("rows")

plt.tight_layout()

In [ ]:
hourly_counts = df.groupby("hour").size().rename("rows")

plt.figure(figsize=(10, 4))
sns.lineplot(x=hourly_counts.index, y=hourly_counts.values, marker="o")
plt.title("Rows by hour")
plt.xlabel("hour")
plt.ylabel("rows")
plt.xticks(range(0, 24))
plt.show()

In [ ]:
sample_size = min(len(df), 20000)
plot_df = df.sample(sample_size, random_state=42) if len(df) > sample_size else df

plt.figure(figsize=(8, 8))
sns.scatterplot(
    data=plot_df,
    x="lon",
    y="lat",
    hue="transport_mode",
    palette="tab10",
    alpha=0.5,
    s=12,
    linewidth=0,
)
plt.title(f"Spatial sample of OpenPFLOW trajectories (n={len(plot_df):,})")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.legend(title="transport_mode", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()